**Assets**
- Total assets: book value of all assets
(i.e. intangible and tangible assets, stock, current and non-currents assets)#
- Total liabilities: sum of current liabilities (i.e. loans and short-term debt, creditors and non-current liabilities (i.e. long-term financial liabilities including borrowing from credit institutions and bonds issued).
- Leverage: ratio of total liabilities to total assets.
  
**Income**
- Operating revenue (turnover): sum of net sales, other operating revenues and stock variations.
- Wage bill: renumeration_employees
- Employment: number of employees on the company’s payroll. 
- Negative turnover values. Turnover is defined as the operating revenue in FAME. In a few cases, some companies report negative turnover values. We flag (but keep) those companies reporting negative turnover values.  
   
**Productivity** 
- GVA (Lars): wage bill + EBITDA
- GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
- Productivity: GVA / employees
- Average wage: wage bill / employees
- Use lns

In [1]:
import ibis
from utils.f_0_dirs import get_data_dirs

old_table_name = "fame_yearly_kp"
new_table_name = "working_yearly"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))

# Reference the existing deflated table
working_yearly_kp = con.table(old_table_name)

# Calculate the new metrics using Ibis lazy evaluation
# We use ibis.ifelse to safely handle natural logarithms of negative or zero GVA
working_yearly_expr = working_yearly_kp.mutate(
    gva = working_yearly_kp.wages + working_yearly_kp.ebitda,
    average_wage = working_yearly_kp.wages / working_yearly_kp.employees
).mutate(
    gva_per_worker = ibis._.gva / working_yearly_kp.employees,
).select(
    "registered_number",
    "year",
    "gva",
    "gva_per_worker",
    "employees",
    "average_wage"
)

print(f"✅ Inserting columns into new '{new_table_name}' table: {working_yearly_expr.columns}")
con.create_table("working_yearly", working_yearly_expr, overwrite=True)

# Verify the final materialized table
final_table = con.table("working_yearly")
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized 'working_yearly' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print("\nHead of working_yearly:")
display(final_table.sample(200 / row_count).execute())

✅ Inserting columns into new 'working_yearly' table: ('registered_number', 'year', 'gva', 'gva_per_worker', 'employees', 'average_wage')
✅ Materialized 'working_yearly' table.
📊 Number of rows: 1,128,490
📊 Number of columns: 6

Head of working_yearly:


,registered_number,year,gva,gva_per_worker,employees,average_wage
0,00174001,2006,832.213515,75.655774,11,49.067321
1,05172628,2006,-460.756593,-46.075659,10,36.130841
2,00293710,2006,-639.068969,-22.823892,28,33.604248
3,00842813,2006,4074.287520,45.269861,90,33.503970
4,01021518,2006,-51297.595504,-32.282942,1589,30.921218
...,...,...,...,...,...,...
186,08403949,2024,1894.190000,30.551452,62,27.966710
187,03231579,2024,11283.078000,156.709417,72,90.873500
188,08899140,2024,2815.665000,24.698816,114,26.146228
189,07781907,2024,7124.773000,274.029731,26,109.193346
